# Лабораторная работа: двумерное преобразование Хаара

**Цель:** реализовать двумерное ортонормированное преобразование Хаара для изображения с тремя уровнями разложения, выполнить обратное преобразование и оценить результат.

В работе используется изображение `monke.jpg`. Преобразование выполняется раздельно по строкам и столбцам. На каждом уровне получаются четыре поддиапазона: `LL`, `LH`, `HL`, `HH`. Следующий уровень применяется только к поддиапазону `LL`.

Формулы для пары пикселей/коэффициентов:

\[
L = \frac{a+b}{\sqrt{2}}, \qquad H = \frac{a-b}{\sqrt{2}}.
\]

Обратные формулы:

\[
a = \frac{L+H}{\sqrt{2}}, \qquad b = \frac{L-H}{\sqrt{2}}.
\]

Используется нормировка `1/sqrt(2)`, поэтому преобразование ортонормированное и сохраняет энергию сигнала.

## 1. Импорт библиотек

In [ ]:
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

## 2. Загрузка и подготовка изображения

Изображение загружается в оттенках серого. Для трёх уровней разложения размер сторон должен делиться на 8. В данном примере приводим изображение к размеру `512 × 512`.

In [ ]:
image_path = "monke.jpg"

img = cv.imread(image_path, cv.IMREAD_GRAYSCALE)
if img is None:
    raise FileNotFoundError(f"Не удалось открыть файл: {image_path}")

img = cv.resize(img, (512, 512), interpolation=cv.INTER_AREA)
img = img.astype(np.float64)

print("Размер изображения:", img.shape)
print("Тип данных:", img.dtype)

plt.figure(figsize=(6, 6))
plt.imshow(img, cmap="gray")
plt.title("Исходное изображение")
plt.axis("off")
plt.show()

## 3. Одноуровневое прямое преобразование Хаара

Сначала преобразуем каждую строку: попарно вычислим низкочастотные (`low`) и высокочастотные (`high`) коэффициенты. Затем аналогичную операцию выполним по столбцам.

В результате получим четыре поддиапазона:
- `LL` — низкие частоты по обоим направлениям;
- `LH` — низкие частоты по строкам и высокие по столбцам;
- `HL` — высокие частоты по строкам и низкие по столбцам;
- `HH` — высокие частоты по обоим направлениям.

In [ ]:
def haar2d_level(image):
    # Преобразование по строкам
    row_low = (image[:, 0::2] + image[:, 1::2]) / np.sqrt(2)
    row_high = (image[:, 0::2] - image[:, 1::2]) / np.sqrt(2)

    # Преобразование по столбцам
    LL = (row_low[0::2, :] + row_low[1::2, :]) / np.sqrt(2)
    LH = (row_low[0::2, :] - row_low[1::2, :]) / np.sqrt(2)
    HL = (row_high[0::2, :] + row_high[1::2, :]) / np.sqrt(2)
    HH = (row_high[0::2, :] - row_high[1::2, :]) / np.sqrt(2)

    return LL, LH, HL, HH

## 4. Трёхуровневое разложение

In [ ]:
# Уровень 1: 512x512 -> поддиапазоны 256x256
LL1, LH1, HL1, HH1 = haar2d_level(img)

# Уровень 2 применяется только к LL1: 256x256 -> 128x128
LL2, LH2, HL2, HH2 = haar2d_level(LL1)

# Уровень 3 применяется только к LL2: 128x128 -> 64x64
LL3, LH3, HL3, HH3 = haar2d_level(LL2)

print("Уровень 1:", LL1.shape, LH1.shape, HL1.shape, HH1.shape)
print("Уровень 2:", LL2.shape, LH2.shape, HL2.shape, HH2.shape)
print("Уровень 3:", LL3.shape, LH3.shape, HL3.shape, HH3.shape)

## 5. Формирование общей карты коэффициентов

Коэффициенты трёх уровней располагаются в одной матрице размером `512 × 512`. В левом верхнем углу находится `LL3`; рядом и ниже расположены детали уровней 3 и 2, а внешние области содержат детали уровня 1.

Для отображения коэффициентов используем логарифмическое преобразование модуля: оно позволяет увидеть как крупные, так и малые значения.

In [ ]:
coeffs = np.zeros_like(img)

# Детали первого уровня
coeffs[0:256, 256:512] = LH1
coeffs[256:512, 0:256] = HL1
coeffs[256:512, 256:512] = HH1

# Детали второго уровня — внутри области LL1
coeffs[0:128, 128:256] = LH2
coeffs[128:256, 0:128] = HL2
coeffs[128:256, 128:256] = HH2

# Детали третьего уровня — внутри области LL2
coeffs[0:64, 64:128] = LH3
coeffs[64:128, 0:64] = HL3
coeffs[64:128, 64:128] = HH3

# Самая грубая аппроксимация
coeffs[0:64, 0:64] = LL3

plt.figure(figsize=(8, 8))
plt.imshow(np.log1p(np.abs(coeffs)), cmap="gray")
plt.title("Карта коэффициентов Хаара (3 уровня, log-масштаб)")
plt.axis("off")
plt.show()

## 6. Обратное преобразование Хаара

Для восстановления сначала объединяем коэффициенты по столбцам, затем по строкам. Это обратная последовательность операций относительно прямого преобразования.

In [ ]:
def ihaar2d_level(LL, LH, HL, HH):
    # Обратное преобразование по столбцам
    row_low = np.zeros((LL.shape[0] * 2, LL.shape[1]), dtype=np.float64)
    row_high = np.zeros_like(row_low)

    row_low[0::2, :] = (LL + LH) / np.sqrt(2)
    row_low[1::2, :] = (LL - LH) / np.sqrt(2)

    row_high[0::2, :] = (HL + HH) / np.sqrt(2)
    row_high[1::2, :] = (HL - HH) / np.sqrt(2)

    # Обратное преобразование по строкам
    reconstructed = np.zeros((row_low.shape[0], row_low.shape[1] * 2),
                             dtype=np.float64)

    reconstructed[:, 0::2] = (row_low + row_high) / np.sqrt(2)
    reconstructed[:, 1::2] = (row_low - row_high) / np.sqrt(2)

    return reconstructed

## 7. Восстановление изображения после трёх уровней

In [ ]:
# Сначала восстанавливаем третий уровень
LL2_rec = ihaar2d_level(LL3, LH3, HL3, HH3)

# Затем второй уровень
LL1_rec = ihaar2d_level(LL2_rec, LH2, HL2, HH2)

# Затем первый уровень
img_rec = ihaar2d_level(LL1_rec, LH1, HL1, HH1)

img_rec = np.clip(img_rec, 0, 255)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(img, cmap="gray", vmin=0, vmax=255)
plt.title("Исходное изображение")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img_rec, cmap="gray", vmin=0, vmax=255)
plt.title("Восстановленное изображение")
plt.axis("off")
plt.show()

## 8. Проверка точности восстановления

Для оценки вычислим среднеквадратичную ошибку (MSE) и пиковое отношение сигнал/шум (PSNR). При отсутствии пороговой обработки ортонормированное преобразование Хаара должно восстанавливать исходные значения с погрешностью вычислений с плавающей точкой.

In [ ]:
mse = np.mean((img - img_rec) ** 2)

if mse == 0:
    psnr = float("inf")
else:
    psnr = 10 * np.log10((255 ** 2) / mse)

max_error = np.max(np.abs(img - img_rec))

print(f"MSE: {mse:.12f}")
print(f"PSNR: {psnr:.6f} dB")
print(f"Максимальная абсолютная ошибка: {max_error:.12e}")

## 9. Дополнительный пример сжатия пороговой обработкой

Само разложение на три уровня — это многоуровневое преобразование. Чтобы продемонстрировать сжатие с потерями, дополнительно обнулим малые по модулю коэффициенты. Порог применяется к карте коэффициентов; затем по изменённым коэффициентам выполним обратное преобразование.

Параметр `threshold` можно менять и наблюдать компромисс между количеством оставшихся коэффициентов и качеством изображения.

In [ ]:
threshold = 12.0

coeffs_thresholded = coeffs.copy()
coeffs_thresholded[np.abs(coeffs_thresholded) < threshold] = 0

# Извлекаем поддиапазоны из пороговой карты
LL3_t = coeffs_thresholded[0:64, 0:64]

LH3_t = coeffs_thresholded[0:64, 64:128]
HL3_t = coeffs_thresholded[64:128, 0:64]
HH3_t = coeffs_thresholded[64:128, 64:128]

LH2_t = coeffs_thresholded[0:128, 128:256]
HL2_t = coeffs_thresholded[128:256, 0:128]
HH2_t = coeffs_thresholded[128:256, 128:256]

LH1_t = coeffs_thresholded[0:256, 256:512]
HL1_t = coeffs_thresholded[256:512, 0:256]
HH1_t = coeffs_thresholded[256:512, 256:512]

# Обратное преобразование
LL2_t = ihaar2d_level(LL3_t, LH3_t, HL3_t, HH3_t)
LL1_t = ihaar2d_level(LL2_t, LH2_t, HL2_t, HH2_t)
img_compressed = ihaar2d_level(LL1_t, LH1_t, HL1_t, HH1_t)
img_compressed = np.clip(img_compressed, 0, 255)

mse_t = np.mean((img - img_compressed) ** 2)
psnr_t = float("inf") if mse_t == 0 else 10 * np.log10((255 ** 2) / mse_t)
zero_percent = 100 * np.count_nonzero(coeffs_thresholded == 0) / coeffs_thresholded.size

print(f"Порог: {threshold}")
print(f"Обнулено коэффициентов: {zero_percent:.2f}%")
print(f"MSE после пороговой обработки: {mse_t:.4f}")
print(f"PSNR после пороговой обработки: {psnr_t:.2f} dB")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(img, cmap="gray", vmin=0, vmax=255)
plt.title("Исходное изображение")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img_compressed, cmap="gray", vmin=0, vmax=255)
plt.title(f"После пороговой обработки (T={threshold})")
plt.axis("off")
plt.show()

## 10. Вывод

В лабораторной работе реализовано двумерное ортонормированное преобразование Хаара для полутонового изображения. Преобразование выполнялось по строкам и столбцам, формируя поддиапазоны `LL`, `LH`, `HL`, `HH`. Последовательно выполнены три уровня разложения: `512 × 512 → 256 × 256 → 128 × 128 → 64 × 64` для аппроксимационной области `LL`.

Реализовано обратное преобразование. Без пороговой обработки изображение восстанавливается практически без ошибки, что подтверждается значениями MSE и PSNR. Дополнительно рассмотрена пороговая обработка коэффициентов: она уменьшает число ненулевых коэффициентов, но приводит к потере части деталей изображения.